# Paquetes de Python

In [4]:
import numpy as np
from typing import Dict
from typing import Optional
from typing import Tuple

ModuleNotFoundError: No module named 'numpy'

# 1. Linear Layer Forward

In [ ]:
def linear_forward(x: np.ndarray, w: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Computes y = x @ w + b
    
    Args:
        x: (N, Din)
        w: (Din, Dout)
        b: (Dout,)
        
    Returns:
        y: (N, Dout)
    """
    # Multiplicación matricial XW y suma del sesgo (broadcasting automático)
    y = x @ w + b
    return y


## Contexto

La capa lineal es la operación más básica de una red neuronal. Lo que hace es tomar un conjunto de entradas y transformarlas mediante una multiplicación matricial seguida de una suma de sesgo. Matemáticamente se expresa como:

$$y = XW + b$$

donde $X \in \mathbb{R}^{N \times D_{in}}$ es la matriz de entrada, $W \in \mathbb{R}^{D_{in} \times D_{out}}$ son los pesos y $b \in \mathbb{R}^{D_{out}}$ es el sesgo.

La intuición detrás de esto es que cada neurona de salida es una combinación lineal ponderada de todas las entradas. Los pesos $W$ determinan qué tan importante es cada feature de entrada para cada neurona de salida, y el sesgo $b$ permite desplazar la activación independientemente de la entrada.

La multiplicación $XW$ proyecta el espacio de entrada de dimensión $D_{in}$ a un nuevo espacio de dimensión $D_{out}$. Esto es lo que le da poder expresivo a la red: apilar capas lineales con activaciones no lineales entre ellas permite aprender transformaciones cada vez más complejas.

Un detalle importante es que $b$ se suma a cada fila de $XW$ gracias al broadcasting de NumPy, es decir, el mismo vector de sesgo se aplica a todas las muestras del batch de forma automática y eficiente, sin necesidad de loops.

# 2. Linear Layer Backward

In [ ]:
def linear_backward(dout: np.ndarray, x: np.ndarray, w: np.ndarray, b: np.ndarray) -> Dict[str, np.ndarray]:
    """
    Computes dx, dw, db for y = x @ w + b.
    
    Args:
        dout: Upstream gradient (N, Dout)
        x: Input (N, Din)
        w: Weights (Din, Dout)
        b: Bias (Dout,)
        
    Returns:
        Dict with "dx", "dw", "db"
    """
    dx =  dout @ w.T
    dw = x.T @ dout
    db = dout.sum(axis=0)
    return {"dx":dx, "dw":dw, "db":db}
    pass

## Contexto

En una capa lineal de una red neuronal, la operación directa se define como:

$$
y = x \cdot W + b
$$

donde:
- $x \in \mathbb{R}^{N \times D_{in}}$: matriz de entrada
- $W \in \mathbb{R}^{D_{in} \times D_{out}}$: matriz de pesos
- $b \in \mathbb{R}^{D_{out}}$: vector de sesgos
- $y \in \mathbb{R}^{N \times D_{out}}$: salida de la capa

Durante el **backpropagation**, se busca calcular los gradientes de la función de pérdida L respecto a cada parámetro: x, W y b.

---

## Derivadas parciales

### 1️. Gradiente respecto a la entrada x
Aplicando la regla de la cadena:

$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial x}
$$

Como $\dfrac{\partial y}{\partial x} = W$, se obtiene:

$$
dx = dout \cdot W^T
$$

donde `dout` representa el gradiente que proviene de la capa siguiente $\frac{\partial L}{\partial y}$.

En código:
```python
dx =  dout @ w.T
```

---

### 2️. Gradiente respecto a los pesos W
De forma análoga:

$$
\frac{\partial L}{\partial W} = x^T \cdot \frac{\partial L}{\partial y}
$$

por lo tanto:

$$
dw = x^T \cdot dout
$$

En código:
```python
dw = x.T @ dout
```

---

### 3️. Gradiente respecto al sesgo b
El sesgo se suma a cada fila de la salida, por lo que su gradiente es la suma de los gradientes de salida a lo largo del batch:

$$
db = \sum_{i=1}^{N} dout_i
$$

En código:  
```python
db = dout.sum(axis=0)
```

#4. ReLU Activation Backward


In [ ]:

def relu_backward(dout: np.ndarray, x: np.ndarray) -> np.ndarray:
   

    # La derivada de ReLU es, a través de una indicadora:
    # 1 si x > 0
    # 0 si x <= 0
    mask = x > 0

    # Aplicamos regla de la cadena.
    dx = dout * mask

    return dx 

## Explicación
Nos interesa saber que tanto cambia el gradiente, si la entrada es positiva , 
se conserva el gradiente pero si es cero o negativa se anula, así vemos que
neuronas aportaron al resultado y cuales no, en el ajuste de parámetros.

# 5 Sigmoid Activation

In [ ]:
def sigmoid_ops(x: np.ndarray, dout: np.ndarray) -> Dict[str, np.ndarray]:
  
    # Forward:
    # La sigmoide transforma cualquier número real en un valor entre 0 y 1.
    # Usamos np.where para evitar problemas numéricos con valores muy grandes.
    out = np.where(
        x >= 0,
        1 / (1 + np.exp(-x)),
        np.exp(x) / (1 + np.exp(x))
    )

    # Backward:
    # La derivada de sigmoid(x) es sigmoid(x) * (1 - sigmoid(x)).
    sigmoid_prime = out * (1 - out)

    # Regla de la cadena.
    dx = dout * sigmoid_prime

    return {
        "out": out,
        "dx": dx
    }



Se usa la función sigmoide tanto atrás como adelante, que se basa en agarrar
 cualquier valor real y mandarlo a un punto entre cero y uno, se defino como
 1 / (1 +np.exp(-x)). Cuando x es muy grande el resultado se acerca a 1 y cuando
 es negativo el resultado se acerca a 0. y después se ve en la parte de 
 backward que se usa paraver como se propaga la gradiente a través del 
 sigmoide.

# 6. Tanh Activation

In [ ]:
def tanh_ops(x: np.ndarray, dout: np.ndarray) -> Dict[str, np.ndarray]:
   """
   Computes tanh forward and backward.
   """
   out = np.tanh(x)
   tanh_prime = 1 - np.tanh(x)**2
   dx = dout*tanh_prime
   return {"out":out, "dx":dx}
   pass

## Contexto


La función de activación tangente hiperbólica se define como:


$$
\tanh(x) = \frac{\sinh(x)}{\cosh(x)} = \frac{e^x - e^{-x}}{e^x + e^{-x}}
$$


Su rango es \(-1, 1\), y se utiliza para normalizar valores en redes neuronales.


---


## Derivación del gradiente
Para el **backward pass**, necesitamos la derivada de $\tanh(x)$ respecto a x:


$$
\frac{d}{dx}\tanh(x) = 1 - \tanh^2(x)
$$


Esto se obtiene aplicando la regla del cociente o usando la identidad hiperbólica:


$$
\cosh^2(x) - \sinh^2(x) = 1
$$


Por tanto:


$$
\frac{d}{dx}\tanh(x) = \frac{\cosh^2(x) - \sinh^2(x)}{\cosh^2(x)} = 1 - \tanh^2(x)
$$


---


## Interpretación en el backward pass
Durante la propagación hacia atrás, el gradiente de la pérdida L respecto a la entrada x se calcula como:


$$
dx = dout \cdot (1 - \tanh^2(x))
$$


donde `dout` es el gradiente que proviene de la capa siguiente.

# 7. MSE Loss

In [ ]:
def mse_loss(y_pred: np.ndarray, y_true: np.ndarray) -> Dict[str, float | np.ndarray]:


    # Diferencia entre la predicción y el valor real.
    error = y_pred - y_true

    # MSE: promedio de los errores al cuadrado.
    loss = np.mean(error ** 2)

    # Número total de elementos.
    num_elements = y_pred.size

    # Gradiente del MSE respecto a y_pred.
    dx = (2 / num_elements) * error

    return {
        "loss": loss,
        "dx": dx
    }

En este ejercicio se implementa la función de pérdida de error cuadrático 
medio. Esta función mide qué tan alejadas están las predicciones del modelo 
respecto a los valores reales. Primero se calcula la diferencia entre la
 predicción y el valor verdadero, luego se eleva al cuadrado y finalmente se
 obtiene el promedio de todos los errores. Además, se calcula el gradiente 
 respecto a las predicciones, el cual indica cómo debe ajustarse la salida
 del modelo para reducir la pérdida. Este gradiente es necesario para la 
 retropropagación, ya que permite actualizar los parámetros de la red neuronal
 en la dirección que disminuye el error.

# 8. Binary Cross Entropy Loss

In [ ]:
def bce_loss(y_pred: np.ndarray, y_true: np.ndarray) -> Dict[str, float | np.ndarray]:
    """
    Computes BCE loss and gradient.
    """
    N = y_pred.size
    eps = 1e-9  # estabilidad numérica

    # Pérdida: promedio del log-verosimilitud negativa (float puro)
    loss = float(-np.mean(
        y_true * np.log(y_pred + eps) + (1 - y_true) * np.log(1 - y_pred + eps)
    ))

    # Gradiente simplificado: (p - y) / N
    dx = (y_pred - y_true) / N

    return {"loss": loss, "dx": dx}


## Contexto

La función de pérdida Binary Cross-Entropy (BCE) se usa en problemas de clasificación binaria donde la salida del modelo es una probabilidad $p \in (0, 1)$, generalmente producida por una sigmoide. Mide qué tan lejos están las probabilidades predichas de las etiquetas reales usando el logaritmo, que penaliza con más fuerza las predicciones muy equivocadas:

$$L = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i\log(p_i) + (1-y_i)\log(1-p_i)\right]$$

Lo que hace esta fórmula es sencillo: cuando la etiqueta real es $y=1$, solo actúa el término $\log(p_i)$, que es mayor (menos negativo) cuanto más cerca esté $p_i$ de 1. Cuando $y=0$, solo actúa $\log(1-p_i)$, que premia predecir probabilidades bajas. En ambos casos, una predicción muy incorrecta como predecir $p \approx 0$ cuando $y=1$ genera una pérdida muy grande porque $\log(0) \to -\infty$.

Para evitar ese problema numérico se agrega $\epsilon = 10^{-9}$ dentro de los logaritmos, lo que garantiza que nunca se evalúe exactamente en cero.

El gradiente respecto a las predicciones se calcula como:

$$\frac{\partial L}{\partial p_i} = \frac{p_i - y_i}{N}$$

Esta forma simplificada surge de combinar la derivada del BCE con la de la sigmoide. Su interpretación es directa: si el modelo predijo $p > y$ el gradiente es positivo y empuja los parámetros en la dirección que baja la predicción. Si predijo $p < y$, el gradiente es negativo y la empuja hacia arriba.

# 10. SGD Optimizer

In [ ]:
def sgd_step(w: np.ndarray, dw: np.ndarray, lr: float) -> np.ndarray:
    """
    Updates w using SGD.
    """
    step = lr*dw
    w_new = w - step

    return w_new
    pass

## Contexto

El **Descenso de Gradiente Estocástico (SGD)** es el algoritmo de optimización más básico y fundamental para entrenar redes neuronales.  
Su objetivo es minimizar una función de pérdida L(w) ajustando los parámetros w en la dirección opuesta al gradiente.

---

## Derivación matemática

La actualización de los parámetros se define como:

$$
w_{\text{new}} = w - \eta \frac{\partial L}{\partial w}
$$

donde:
- w: vector de pesos actuales  
- $\eta$: tasa de aprendizaje (learning rate)  
- $\frac{\partial L}{\partial w}$: gradiente de la pérdida respecto a los pesos  

En el caso estocástico, el gradiente se calcula sobre un **subconjunto (batch)** de los datos, lo que introduce cierta variabilidad pero acelera el entrenamiento.

---

## Interpretación geométrica
El gradiente $\frac{\partial L}{\partial w}$ apunta hacia la dirección de **mayor aumento** de la pérdida.  
Por tanto, el paso de actualización $ -\eta \frac{\partial L}{\partial w} $ mueve los parámetros **en sentido contrario**, buscando el mínimo local.

$$
\text{SGD step} = -\eta \cdot \text{gradient}
$$

Cada iteración reduce la pérdida L(w) hasta converger a un valor mínimo.

## 11. Momentum Optimizer

In [ ]:
def momentum_step(
    w: np.ndarray,
    dw: np.ndarray,
    v: np.ndarray,
    lr: float,
    momentum: float
    )-> Tuple[np.ndarray, np.ndarray]:
    """
    Performs SGD with momentum update.
    Returns (w_new, v_new).
    """

    # Actualizamos la velocidad acumulando parte de la velocidad anterior
    # y sumando el gradiente actual.
    v_new = momentum * v + dw

    # Actualizamos los pesos usando la nueva velocidad.
    w_new = w - lr * v_new

    return w_new, v_new


En este ejercicio se implementa una actualización de pesos usando descenso por 
gradiente con momentum. La idea del momentum es acumular parte de las 
actualizaciones anteriores mediante una variable de velocidad. Esto permite que
el entrenamiento sea más estable, ya que el modelo no depende únicamente del 
gradiente calculado en el paso actual. Primero se actualiza la velocidad 
combinando la velocidad anterior con el gradiente actual, y luego se actualizan
 los pesos restando la tasa de aprendizaje multiplicada por esa nueva velocidad
 En términos prácticos, momentum ayuda a suavizar el movimiento de los
 parámetros durante el entrenamiento de la red neuronal.


# 12. RMSProp Optimizer

In [ ]:
def rmsprop_step(
    w: np.ndarray,
    dw: np.ndarray,
    cache: np.ndarray,
    lr: float,
    decay: float,
    eps: float = 1e-8
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Performs one RMSProp update step.

    Returns:
        w_new: updated weights
        cache_new: updated moving average of squared gradients
    """
    # Paso 1: actualizar la media móvil de gradientes al cuadrado
    cache_new = decay * cache + (1 - decay) * dw ** 2

    # Paso 2: tasa de aprendizaje adaptativa
    adaptive_lr = lr / (np.sqrt(cache_new) + eps)

    # Paso 3: actualizar pesos
    w_new = w - adaptive_lr * dw

    return w_new, cache_new

## Contexto

RMSProp es una mejora sobre el descenso de gradiente estándar que adapta la tasa de aprendizaje de forma individual para cada parámetro. La idea central es que los parámetros cuyos gradientes han sido grandes en el pasado deberían recibir actualizaciones más pequeñas, y viceversa. Esto se logra manteniendo una media móvil de los gradientes al cuadrado:

$$s_{t+1} = \beta \cdot s_t + (1 - \beta) \cdot (\nabla L)^2$$

$$w_{t+1} = w_t - \frac{\eta}{\sqrt{s_{t+1} + \epsilon}} \cdot \nabla L$$

El término $s_{t+1}$ acumula una historia reciente de cuán grandes han sido los gradientes. Al dividir la tasa de aprendizaje $\eta$ por $\sqrt{s_{t+1}}$, los parámetros con gradientes históricamente grandes reciben pasos más pequeños, y los que tienen gradientes pequeños reciben pasos más grandes.

El parámetro $\beta$ (decay) controla qué tan larga es esa memoria: valores cercanos a 1 como 0.99 dan más peso al pasado y producen una adaptación más suave, mientras que valores más bajos como 0.9 reaccionan más rápido a cambios recientes en los gradientes.

El $\epsilon$ en el denominador cumple dos funciones: evitar la división por cero cuando $s \approx 0$, y estabilizar el comportamiento cuando los gradientes son muy pequeños al inicio del entrenamiento.

# 15. Weight Initialization (Kaiming/He)

In [ ]:
def kaiming_init(shape: tuple) -> np.ndarray:
    """
    Kaiming/He normal initialization.
    """
    fan_in = shape[0]
    fan_out = shape[1]

    std = np.sqrt(2.0 / fan_in)

    weights = np.random.normal(
        loc=0.0,
        scale=std,
        size=shape
    )

    return weights
    pass

## Contexto
La **inicialización de Kaiming (He)** se diseñó específicamente para redes neuronales que usan la función de activación ReLU.  
Su objetivo es mantener la **varianza de las activaciones constante** a lo largo de las capas, evitando que los gradientes se desvanezcan o exploten.

---

## Derivación matemática

Consideremos una capa lineal con pesos W y entradas x:

$$
y = W \cdot x
$$

Queremos que la varianza de la salida y sea igual a la varianza de la entrada x:

$$
Var(y) = Var(x)
$$

Si los pesos W se inicializan con media cero y varianza Var(W), entonces:

$$
Var(y) = Var(W) \cdot Var(x) \cdot n
$$

donde n es el número de entradas (fan\_in).

Para mantener Var(y) = Var(x), se requiere:

$$
Var(W) = \frac{1}{n}
$$

Sin embargo, cuando se usa **ReLU**, aproximadamente la mitad de las activaciones se anulan (porque ReLU(x) = 0 para x < 0).  
Por tanto, la varianza efectiva se reduce a la mitad, y se compensa multiplicando por 2:

$$
Var(W) = \frac{2}{n}
$$

Esto se traduce en inicializar los pesos con una distribución normal de media 0 y desviación estándar:

$$
\sigma = \sqrt{\frac{2}{n}}
$$

# 16. Dropout Forward (Inverted)

In [3]:
def dropout_forward(
    x: np.ndarray,
    p: float,
    train: bool = True,
    seed: Optional[int] = None
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """
    Inverted dropout forward. Returns (out, mask).
    """
    if not train:
        return x, None

    if seed is not None:
        np.random.seed(seed)

    # Generar máscara binaria: 1 = mantener, 0 = desactivar
    mask = (np.random.rand(*x.shape) >= p).astype(x.dtype)

    # Caso especial: p=1: todas las neuronas desactivadas, salida cero
    if p == 1.0:
        return np.zeros_like(x), mask

    # Aplicar máscara y escalar (dropout invertido)
    out = (mask * x) / (1 - p)

    return out, mask

NameError: name 'np' is not defined

## Contexto

El Dropout es una técnica de regularización que consiste en apagar aleatoriamente un porcentaje de las neuronas durante el entrenamiento. La intuición es forzar a la red a no depender demasiado de ninguna neurona en particular, lo que reduce el sobreajuste y hace el modelo más robusto.

Durante el forward pass de entrenamiento, se genera una máscara binaria aleatoria donde cada elemento tiene probabilidad $p$ de ser cero (neurona apagada) y $1-p$ de ser uno (neurona activa). Luego se aplica la variante invertida:

$$\text{out} = \frac{\text{mask}}{1 - p} \odot x, \quad \text{mask} \sim \text{Bernoulli}(1 - p)$$

El factor de escala $\frac{1}{1-p}$ es lo importante del dropout invertido. Como en promedio solo quedan activas una fracción $1-p$ de las neuronas, sin ese factor la magnitud de la salida sería menor que la entrada original. Al escalar durante el entrenamiento se garantiza que el valor esperado de la salida sea igual al de la entrada:

$$\mathbb{E}[\text{out}] = \mathbb{E}[x]$$

Esto tiene una ventaja importante: durante la evaluación no hay que hacer ningún ajuste, simplemente se pasa la entrada directamente sin modificar. La máscara se guarda y se reutiliza en el backward pass, ya que los gradientes solo deben fluir por las neuronas que estuvieron activas en el forward.

# 17. Dropout Backward

In [ ]:
def dropout_backward(dout: np.ndarray, mask: Optional[np.ndarray], p: float, train: bool = True) -> np.ndarray:
    """
    Backward pass for inverted dropout.
    """
    if train:
        masked_grad = mask * dout
        dx = masked_grad / (1 - p)
    else:
        dx = dout

    return dx
    pass

## Contexto
El **Dropout** es una técnica de regularización que consiste en desactivar aleatoriamente un conjunto de neuronas durante el entrenamiento para evitar el sobreajuste (*overfitting*).  
Durante la propagación hacia atrás (**backpropagation**), los gradientes solo fluyen a través de las neuronas que **no fueron desactivadas**.

---

## Definición del proceso
Durante el **forward pass**, se aplica una máscara binaria m que indica qué neuronas se mantienen activas:

$$
y = \frac{m \cdot x}{1 - p}
$$

donde:
- x: entrada de la capa  
- m: máscara binaria (1 = activa, 0 = desactivada)  
- p: probabilidad de dropout  
- 1 - p: factor de escala para mantener la expectativa constante  

---

## Derivación del backward pass
En el **backward pass**, el gradiente de la pérdida L respecto a la entrada x se calcula como:

$$
\frac{\partial L}{\partial x} = \frac{m}{1 - p} \cdot \frac{\partial L}{\partial y}
$$

Esto significa que:
- Solo las neuronas activas (m = 1) reciben gradiente.  
- Las neuronas desactivadas (m = 0) no contribuyen al gradiente.  
- El factor $\frac{1}{1 - p}$ mantiene la escala esperada de las activaciones.

# 18.  Batch Normalization Forward

In [ ]:
def batchnorm_forward(
    x: np.ndarray,
    gamma: np.ndarray,
    beta: np.ndarray,
    eps: float = 1e-5,
    momentum: float = 0.9,
    running_mean: Optional[np.ndarray] = None,
    running_var: Optional[np.ndarray] = None,
    train: bool = True
    ):
   

    # Si no existen estadísticas acumuladas, se inicializan en cero.
    if running_mean is None:
        running_mean = np.zeros(x.shape[1])

    if running_var is None:
        running_var = np.zeros(x.shape[1])

    if train:
        # Media y varianza del batch actual, calculadas por columna.
        mean = np.mean(x, axis=0)
        var = np.mean((x - mean) ** 2, axis=0)

        # Normalización del batch.
        x_norm = (x - mean) / np.sqrt(var + eps)

        # Escalamiento y desplazamiento aprendibles.
        out = gamma * x_norm + beta

        # Actualización de estadísticas acumuladas.
        running_mean_new = momentum * running_mean + (1 - momentum) * mean
        running_var_new = momentum * running_var + (1 - momentum) * var

        # Guardamos información útil para backward.
        cache = {
            "x": x,
            "x_norm": x_norm,
            "mean": mean,
            "var": var,
            "gamma": gamma,
            "eps": eps
        }

    else:
        # En evaluación no usamos la media y varianza del batch,
        # sino las estadísticas acumuladas durante entrenamiento.
        x_norm = (x - running_mean) / np.sqrt(running_var + eps)

        out = gamma * x_norm + beta

        running_mean_new = running_mean
        running_var_new = running_var

        cache = {
            "x": x,
            "x_norm": x_norm,
            "mean": running_mean,
            "var": running_var,
            "gamma": gamma,
            "eps": eps
        }

    return out, cache, running_mean_new, running_var_new

En este ejercicio se implementa la pasada hacia adelante de Batch 
Normalization. Esta técnica normaliza las activaciones de una capa usando la 
 y la varianza del batch, con el objetivo de estabilizar el entrenamiento de 
 la red neuronal. Después de normalizar, se aplican dos parámetros aprendibles,
 gamma y beta, que permiten ajustar la escala y el desplazamiento de los datos 
 normalizados. Durante el entrenamiento se calculan las estadísticas del batch 
 actual y se actualizan estadísticas acumuladas. Durante evaluación, en cambio,
 se usan esas estadísticas acumuladas para mantener un comportamiento estable 
 del modelo.

# 20. Layer Normalization Forward & Backward

In [ ]:
def layernorm(
    x: np.ndarray,
    gamma: np.ndarray,
    beta: np.ndarray,
    eps: float = 1e-5
) -> Tuple[np.ndarray, Tuple]:
    """
    LayerNorm forward. Returns (out, cache).
    """
    # Estadísticas por muestra (a lo largo de las features)
    mean = x.mean(axis=1, keepdims=True)          # (N, 1)
    var  = x.var(axis=1, keepdims=True)            # (N, 1)

    # Normalizar
    x_norm = (x - mean) / np.sqrt(var + eps)       # (N, D)

    # Escalar y desplazar con parámetros aprendibles
    out = gamma * x_norm + beta                    # (N, D)

    cache = (x, x_norm, mean, var, gamma, eps)
    return out, cache


def layernorm_backward(dout: np.ndarray, cache: Tuple) -> Dict[str, np.ndarray]:
    """
    LayerNorm backward. Returns dict with dx, dgamma, dbeta.
    """
    x, x_norm, mean, var, gamma, eps = cache
    N, D = x.shape

    # Gradientes de gamma y beta (sumados sobre el batch)
    dbeta  = dout.sum(axis=0)                      # (D,)
    dgamma = (dout * x_norm).sum(axis=0)           # (D,)

    # Gradiente respecto a x_norm
    dx_norm = dout * gamma                         # (N, D)

    # Gradiente respecto a la varianza
    std_inv = 1.0 / np.sqrt(var + eps)             # (N, 1)
    dvar = (-0.5 * (dx_norm * (x - mean)).sum(axis=1, keepdims=True)
            * std_inv ** 3)                        # (N, 1)

    # Gradiente respecto a la media
    dmean = (-std_inv * dx_norm).sum(axis=1, keepdims=True) \
            + dvar * (-2.0 / D) * (x - mean).sum(axis=1, keepdims=True)  # (N, 1)

    # Gradiente respecto a x
    dx = (dx_norm * std_inv
          + dvar * 2.0 * (x - mean) / D
          + dmean / D)                            # (N, D)

    return {"dx": dx, "dgamma": dgamma, "dbeta": dbeta}

## Contexto

Layer Normalization normaliza las activaciones de cada muestra del batch de forma independiente, a lo largo de la dimensión de features. A diferencia de Batch Normalization que normaliza a lo largo del batch, LayerNorm no depende del tamaño del batch, lo que lo hace ideal para modelos de lenguaje y transformers donde el batch puede ser pequeño o variable.

Para cada muestra $i$ se calculan su media y varianza propias a lo largo de sus features:

$$\mu_i = \frac{1}{D}\sum_{j=1}^{D} x_{ij}, \qquad \sigma_i^2 = \frac{1}{D}\sum_{j=1}^{D}(x_{ij} - \mu_i)^2$$

Luego se normaliza cada feature de esa muestra:

$$\hat{x}_{ij} = \frac{x_{ij} - \mu_i}{\sqrt{\sigma_i^2 + \epsilon}}$$

Y finalmente se aplican los parámetros aprendibles $\gamma$ y $\beta$ que le devuelven capacidad expresiva a la red:

$$y_{ij} = \gamma_j \hat{x}_{ij} + \beta_j$$

Sin $\gamma$ y $\beta$, la normalización forzaría siempre media cero y varianza unitaria, lo que podría ser demasiado restrictivo. Con ellos, la red aprende cuál es la escala y desplazamiento óptimos para cada feature.

En el backward pass se propagan los gradientes a través de la normalización usando la regla de la cadena. Primero se calculan $d\beta$ y $d\gamma$ sumando sobre el batch, luego se obtiene el gradiente respecto a $\hat{x}$, y finalmente se propaga hasta la entrada $x$ pasando por la varianza y la media. Es importante usar `keepdims=True` al calcular las estadísticas para que el broadcasting funcione correctamente entre tensores de forma $(N, 1)$ y $(N, D)$.